# Datbase Generation: Multiple cores

In [1]:
import numpy as np
np.infty = np.inf
import sys
import numpy as np
import cantera as ct
import matplotlib.pyplot as plt
import itertools
import scipy.integrate
import ctypes as xt
import os
from ideal_reactor_models import customESC_BM
from ideal_reactor_models import customPFR
import pandas as pd
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

In [2]:
def calc_conversion(gas,states, reactant):
    return 100.*(states.Y[0,gas.species_index(reactant)]-states.Y[:,gas.species_index(reactant)])/states.Y[0,gas.species_index(reactant)]
def calc_yield(gas,states, reactant, product):
    return 100.*(states.Y[:,gas.species_index(product)]-states.Y[0,gas.species_index(product)])/states.Y[0,gas.species_index(reactant)]
def calc_selectivity(gas,states, reactant, product):
    return 100.*(states.Y[:,gas.species_index(product)]-states.Y[0,gas.species_index(product)])/(states.Y[0,gas.species_index(reactant)]-states.Y[:,gas.species_index(reactant)])

In [3]:
def CRACKSIM_rates_DLL(gas):
    # CRACKSIM_rates_C call the rates function in CRACKSIM.DLL
    # concentration bassis 

    # initialize temperature and concentrations
    T= gas.T     # [K]
    C_point=gas.concentrations #[mol/l]
    status = 0              

    # copy values to Ctypes to be used as arguements for the Fortran DLL
    C_point= gas.concentrations.ctypes
    T_point = xt.byref(xt.c_double(T)) 
    status = xt.pointer(xt.c_int(status))


    # initalize a ctype pointer to be used as storage for the calculated rates 
    R_point = (xt.c_double*gas.n_species)()     # mol/(s.L)
  
    _ = fortlib.NetRates_C(T_point,C_point,R_point,status)  # function call of CRACKSIM DLL
    
    # convert Ctype to python array
    rates=np.ctypeslib.as_array(R_point)
    rates=rates         #[mol/l/s]
    return rates
    #return rates

In [5]:
# initialize kinetics 
fortlib = xt.CDLL(r"C:\Users\mbonheur\OneDrive - UGent\Documenten\GitHub\Thesis_Louis\SA_CRACKSIM.dll") #change this to the location where you stored the dll
status = 0
option = (xt.c_int*20)()
# please note that in python arrays possitions starts counting from 0 
            # option(1) : 
            #      0 use full network
            #      1 use reduced network based on compossition file 
            #      2 use only betanetwork
option[0]= 1
status = xt.pointer(xt.c_int(status)) # setup the pointer to the correct data structure
_ = fortlib.Initialise_CRACKSIM(status,option)            # call the function
status = status[0]
if status==1:
    print("Kinetics and Thermo were read succesfuly")
else:
    print("Errors while reading Kinetics and Thermo")

Kinetics and Thermo were read succesfuly


## Foutmelding NASA polynomes

In [8]:
#Included to suppress the error warnings related to the NASA polynomes
ct.suppress_thermo_warnings()       # currently an issue with Nasapolynomials 
# convert the chem.inp file created by the DLL to yaml file

!ck2yaml --input=chem.inp --transport=transport_chemkin.DAT --permissive > C2KYAML_log.txt

C:\Users\mbonheur\Anaconda3\envs\Cantera_env_3-1\lib\site-packages\cantera\ck2yaml.py:2344: UserWarning: NasaPoly2::validate: 
For species CO, discontinuity in h/RT detected at Tmid = 1400
	Value computed using low-temperature polynomial:  -6.48615241904762
	Value computed using high-temperature polynomial: -6.431806107428571

  gas = Solution(out_name)
C:\Users\mbonheur\Anaconda3\envs\Cantera_env_3-1\lib\site-packages\cantera\ck2yaml.py:2344: UserWarning: NasaPoly2::validate: 
For species CO2, discontinuity in h/RT detected at Tmid = 1400
	Value computed using low-temperature polynomial:  -29.0113435047619
	Value computed using high-temperature polynomial: -28.977686331657146

  gas = Solution(out_name)
C:\Users\mbonheur\Anaconda3\envs\Cantera_env_3-1\lib\site-packages\cantera\ck2yaml.py:2344: UserWarning: NasaPoly2::validate: 
For species NEOC5, discontinuity in cp/R detected at Tmid = 1400
	Value computed using low-temperature polynomial:  42.78292259039995
	Value computed using hig

## Verder met code

In [13]:
reac_mech_DLL = 'chem.yaml'
gas_id = 'gas'
gas = ct.Solution(reac_mech_DLL)
print('Gas mechanism contains {} species and {} reactions'.format(gas.n_species, gas.n_reactions))

Gas mechanism contains 239 species and 0 reactions


In [15]:
massflow = [80,85] #np.linspace(75,100,2) #80.4367 #kg/s
diam = 0.5 #m
T_in = [650+273,850+273] #np.linspace(550+273.15,1250+273.15,3)#650 + 273.15 #K
P_in = [1.5e+5,2.0e+5] #np.linspace(1.0e+5,3.5e+5,3)#2.0e+5 #Pa
Y_in = {'C2H6':0.77,'H2O':0.23} #massfractions inlet stream
N_steps = 100 #Aantal integratiestappen
H_input = [0,25e+6,50e+6] #np.linspace(0,50e+6,3) #50000000 #W/m2, discrete heat input
NS = [5,15,30] #np.linspace(5,30,3,dtype=int) #20 # Aantal heat inputs, (aantal passages door de blades vd turboreactor)
H2O_fraction = [0.3,0.5] #np.linspace(0.3,0.5,3)

In [19]:
cases = list(itertools.product(H_input, NS, massflow, T_in, P_in, H2O_fraction))

In [72]:
def make_hf(z_prof, hf_prof):
    def hf_func(z):
        return np.interp(z, z_prof, hf_prof)
    return hf_func

In [87]:
# TEST NIEUWE CONSTRUCTIE HEAT PROFILE

# z_prof = np.linspace(0,10.5,100).tolist() #Axiale afstand, m
# hf_prof = [] #Heatflux, W/m2
# for i in range(len(z_prof)):
#     if i%(len(z_prof)//10) == 0:
#         hf_prof.append(20e6) #W/m2
#     if i%(len(z_prof)//10) != 0:
#         hf_prof.append(0)
# length = z_prof[-1] - z_prof[0] #m
# hf_func = make_hf(z_prof, hf_prof)
# hf = ct.Func1(hf_func)   # ct.Func1(lambda z: np.interp(z,z_prof,hf_prof))
# y = []
# x = []
# for i in z_prof:
#     y.append(hf(i))
#     x.append(i)
# #y.append(0)
# plt.figure()
# plt.plot(x,y)
# plt.show()

In [31]:
print(multiprocessing.cpu_count())

8


In [99]:
def run_case(argument):
    # argument: tuple (H, N, M, T, P, X)
    try:
        reac_mech_DLL = 'chem.yaml'
        gas_id = 'gas'
        gas = ct.Solution(reac_mech_DLL)
    
        
        H, N, M, T, P, X = argument
    
        z_prof = np.linspace(0,10.5,100).tolist() #Axiale afstand, m
        hf_prof = [] #Heatflux, W/m2
        for i in range(len(z_prof)):
            if i%(len(z_prof)//N) == 0:
                hf_prof.append(H) #W/m2
            if i%(len(z_prof)//N) != 0:
                hf_prof.append(0)
        length = z_prof[-1] - z_prof[0] #m
        hf_func = make_hf(z_prof, hf_prof)
        hf = ct.Func1(hf_func)   # ct.Func1(lambda z: np.interp(z,z_prof,hf_prof))
        y = []
        for i in z_prof:
            y.append(hf(i))
        y.append(0)
        hf, y = heat_profile(H,N)
        PFR_calc = customPFR(reac_mech_DLL,gas_id,M,diam,CRACKSIM_rates_DLL,energy_type = 'heat-flux-profile',heat_flux = hf, U = None, Tr = None, friction_factors = None)
        PFR_calc.gas.TPY = T, P, {'C2H6':(1.0 - X), 'H2O':X}
        states_PFR_calc,rates_PFR_calc,enth_PFR_calc = PFR_calc.solve(length,100)
    
        df_Y = pd.DataFrame(states_PFR_calc.Y, columns = states_PFR_calc.species_names)
        df_Y.rename(columns = {name: f"Y_{name}" for name in df_Y.columns},inplace = True)
        df_rr = pd.DataFrame(rates_PFR_calc,columns = states_PFR_calc.species_names)
        df_rr.rename(columns = {name: f"R_{name}" for name in df_rr.columns}, inplace = True)
        df_species = pd.concat([df_Y, df_rr], axis = 1)
        df_species['T [K]'] = states_PFR_calc.T
        df_species['P [Pa]'] = states_PFR_calc.P
        df_species['S Energy [J/s/m3]'] = states_PFR_calc.energy_source_term
        df_species['Mass flow [kg/s]'] = states_PFR_calc.mass_flow
        df_species['Heat input [W/m^2]'] = y
    
        return df_species
    except Exception as e:
        import traceback
        print(f"Fout in case {argument}: {e}")
        traceback.print_exc()
        return None

In [107]:
excel_database = "Database_TEST11.xlsx"

if __name__ == '__main__':
    results = []
    with ProcessPoolExecutor(max_workers = multiprocessing.cpu_count()) as executor:
        for i, df in enumerate(executor.map(run_case, cases), start = 1):
            if df is None:
                continue #overslaan als run_case faalt
            mode = 'w' if i == 1 else 'a'
            df.to_excel(excel_database,sheet_name = str(i), engine = 'openpyxl', mode = mode, index = False)

            

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [109]:
run_case(cases[-1])

,Y_H2,Y_CH4,Y_C2H6,Y_C3H8,Y_IC4H10,Y_NC4H10,Y_H2O,Y_CO,Y_CO2,Y_NEOC5,...,R_VC4H5-1.,R_VC4H5-2.,R_CPD.,R__1m-napht,R__1m-anthr,T [K],P [Pa],S Energy [J/s/m3],Mass flow [kg/s],Heat input [W/m^2]
0,0.000000,0.000000,0.500000,0.000000,0.000000e+00,0.000000,0.5,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1123.000000,200000.0,-1.072467e+06,85,50000000.0
1,0.000254,0.000025,0.496137,0.000007,5.203227e-12,0.000017,0.5,1.828987e-10,3.516999e-14,2.957051e-19,...,5.460454e-10,1.592313e-07,3.054339e-11,1.783232e-30,3.330082e-46,1132.089129,200000.0,-1.140975e+08,85,0.0
2,0.000603,0.000049,0.490875,0.000016,2.784678e-11,0.000048,0.5,7.330159e-10,2.852863e-13,4.334756e-18,...,2.037163e-09,1.077007e-06,1.511732e-09,1.233904e-25,1.556064e-39,1124.701384,200000.0,-9.014631e+07,85,0.0
3,0.000901,0.000070,0.486363,0.000023,6.063003e-11,0.000074,0.5,1.576370e-09,9.203627e-13,1.683088e-17,...,8.179241e-09,3.112005e-06,1.027502e-08,2.528550e-23,2.596242e-36,1132.028220,200000.0,-9.798471e+07,85,50000000.0
4,0.001297,0.000101,0.480383,0.000035,1.358736e-10,0.000108,0.5,2.989150e-09,2.382732e-12,6.896591e-17,...,1.575528e-08,8.557653e-06,6.078191e-08,2.083403e-21,1.069471e-33,1138.968460,200000.0,-1.218634e+08,85,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,0.032463,0.025512,0.031329,0.000798,1.400870e-07,0.000134,0.5,3.751211e-05,1.491282e-06,6.419559e-10,...,-1.029490e-04,-4.132026e-03,1.246462e-02,7.529615e-05,1.314543e-08,1345.567497,200000.0,-6.854138e+07,85,50000000.0
97,0.032689,0.026182,0.029229,0.000756,1.288509e-07,0.000121,0.5,3.975175e-05,1.622849e-06,6.324991e-10,...,3.670556e-04,9.993511e-03,6.263130e-02,1.323651e-04,2.395620e-08,1355.300400,200000.0,-8.102002e+07,85,0.0
98,0.032963,0.026985,0.026785,0.000708,1.158073e-07,0.000105,0.5,4.231963e-05,1.778452e-06,6.186048e-10,...,-1.617180e-04,-4.943369e-03,2.547609e-02,1.323783e-04,2.649809e-08,1362.192263,200000.0,-8.609106e+07,85,0.0
99,0.033204,0.027713,0.024739,0.000665,1.049494e-07,0.000093,0.5,4.491160e-05,1.940705e-06,6.035432e-10,...,-1.235583e-04,-4.405023e-03,1.003475e-02,1.004242e-04,2.338050e-08,1357.005695,200000.0,-6.798342e+07,85,50000000.0
